# Route Resilience — Road Segmentation Training (Kaggle)

## ⚠️ Required Kaggle Settings (set BEFORE running any cell)

1. **Settings → Accelerator → GPU T4 x2** (recommended) or **GPU P100** (simpler single-GPU).
   - **GPU T4 x2**: Two T4s (2×16 GB VRAM). PyTorch will use the first GPU by default; enable `nn.DataParallel` if you want to split batches across both.
   - **GPU P100**: Single GPU (16 GB VRAM). Simpler — no multi-GPU code needed. Plenty fast for a ResNet34-backbone U-Net at 512×512 tiles.
   - **Without this setting, training silently falls back to CPU (~50× slower).**

2. **Settings → Internet → On** (REQUIRED). Internet access must be enabled to download packages like `segmentation-models-pytorch` that are not pre-installed in the base environment.

## Attached Datasets
- **DeepGlobe Road Extraction** — RGB satellite tiles + RGB class masks + `class_dict.csv`
- **HybridSAR Road Dataset (HSRD)** — SAR imagery (deferred to a future training run)

## Step 0 — Discover actual dataset folder names
Run this cell **first**. Copy the exact folder name(s) printed below into the `DEEPGLOBE_DIR` / `HSRD_DIR` constants in the next cell.

In [ ]:
import os
print('Datasets mounted under /kaggle/input:')
for entry in sorted(os.listdir('/kaggle/input')):
    full = os.path.join('/kaggle/input', entry)
    n_items = len(os.listdir(full)) if os.path.isdir(full) else 0
    print(f'  {entry}/   ({n_items} items)')

In [ ]:
# ============================================================
# CONFIG — Smart Path Auto-Detection
# ============================================================
import os

# ---- Smart Auto-Detection of Dataset Directories ----
DEEPGLOBE_DIR = None
HSRD_DIR = None

for root, dirs, files in os.walk('/kaggle/input'):
    # Auto-detect DeepGlobe by class_dict.csv and metadata.csv
    if 'class_dict.csv' in files and 'metadata.csv' in files:
        DEEPGLOBE_DIR = root
    # Auto-detect HSRD by unique folders
    if any(d in dirs for d in ['DG-SAR', 'SN3-SAR', 'SN6R']):
        HSRD_DIR = root

# Fallback to defaults if auto-detection fails
if DEEPGLOBE_DIR is None:
    DEEPGLOBE_DIR = '/kaggle/input/deepglobe-road-extraction-dataset'
    print(f"⚠️ DeepGlobe not auto-detected, using fallback: {DEEPGLOBE_DIR}")
else:
    print(f"✅ DeepGlobe auto-detected at: {DEEPGLOBE_DIR}")

if HSRD_DIR is None:
    HSRD_DIR = '/kaggle/input/the-hybridsar-road-dataset-hsrd'
    print(f"⚠️ HSRD not auto-detected, using fallback: {HSRD_DIR}")
else:
    print(f"✅ HSRD auto-detected at: {HSRD_DIR}")

# ---- Training hyperparameters ----
TILE_SIZE = 512
BATCH_SIZE = 8          # Reduce to 4 if OOM on T4 (16 GB VRAM)
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4

# Model: DeepGlobe = RGB = in_channels=3, no change needed
ARCHITECTURE = 'Unet'   # Options: 'Unet', 'UnetPlusPlus', 'DeepLabV3Plus'
ENCODER = 'resnet34'
IN_CHANNELS = 3         # DeepGlobe is standard RGB

# ---- Output paths ----
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# CONDITIONAL INSTALLS
# Kaggle pre-installs PyTorch + CUDA — do NOT reinstall torch.
# Only install packages missing from the Kaggle base image.
# ============================================================

def _ensure(pkg_import, pip_name=None):
    try:
        __import__(pkg_import)
    except ImportError:
        import subprocess, sys
        pkg = pip_name or pkg_import
        print(f"Installing {pkg}...")
        # Base install command
        cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check']
        
        # Handle PEP 668 Externally Managed Environments on Python 3.12+
        if sys.version_info >= (3, 12):
            cmd.append('--break-system-packages')
            
        cmd.append(pkg)
        
        # Run and capture output to print exact error on failure
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print("\n" + "!"*60)
            print(f"CRITICAL ERROR: Failed to install '{pkg}'.")
            print("--- PIP STDOUT ---")
            print(result.stdout)
            print("--- PIP STDERR ---")
            print(result.stderr)
            print("\nPOSSIBLE SOLUTIONS:")
            print("1. Turn on Internet in your Kaggle Notebook Settings (right sidebar -> Settings -> Internet -> On).")
            print("2. Make sure you are using a GPU accelerator (T4 or P100).")
            print("!"*60 + "\n")
            raise RuntimeError(f"Failed to install {pkg}: {result.stderr}")
        else:
            print(f"Successfully installed {pkg}.")

_ensure('segmentation_models_pytorch', 'segmentation-models-pytorch')
_ensure('albumentations')
_ensure('osmnx')
_ensure('rasterio')
print('All packages ready.')

In [ ]:
# ============================================================
# IMPORT PROJECT MODULES
# ============================================================
import os
import sys

# Auto-detect exact source directory (handles nested, flat, and user-specific layouts)
possible_paths = [
    '/kaggle/input/datasets/satyveeryadav/route-resillence-src/route-resillence-src',
    '/kaggle/input/route-resillence-src/route-resillence-src',
    '/kaggle/input/route-resillence-src'
]
RAW_SRC_PATH = None
for path in possible_paths:
    if os.path.exists(os.path.join(path, 'data_pipeline.py')):
        RAW_SRC_PATH = path
        break

if RAW_SRC_PATH is not None:
    print(f"✅ Raw source path detected: {RAW_SRC_PATH}")
    
    # Copy to writable /kaggle/working to patch bugs
    SRC_DATASET_PATH = '/kaggle/working/route-resillence-src'
    import shutil
    if os.path.exists(SRC_DATASET_PATH):
        shutil.rmtree(SRC_DATASET_PATH)
    shutil.copytree(RAW_SRC_PATH, SRC_DATASET_PATH)
    print(f"✅ Copied source files to writable directory: {SRC_DATASET_PATH}")
    
    # Patch target float casting bugs in segmentation_train.py on disk
    train_py = os.path.join(SRC_DATASET_PATH, 'segmentation_train.py')
    with open(train_py, 'r', encoding='utf-8') as f:
        code = f.read()
        
    if "targets = targets.float()" not in code:
        # 1. Cast targets to float in DiceBCELoss.forward
        code = code.replace(
            "    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:",
            "    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\n        targets = targets.float()"
        )
        # 2. Cast targets to float in compute_metrics
        code = code.replace(
            "    probs = torch.sigmoid(preds)\n    binary = (probs > threshold).float()",
            "    targets = targets.float()\n    probs = torch.sigmoid(preds)\n    binary = (probs > threshold).float()"
        )
        
    with open(train_py, 'w', encoding='utf-8') as f:
        f.write(code)
    print("✅ Patched segmentation_train.py on disk successfully.")
    
    # Direct python to load from the patched writable directory
    if SRC_DATASET_PATH not in sys.path:
        sys.path.insert(0, SRC_DATASET_PATH)
    else:
        sys.path.remove(SRC_DATASET_PATH)
        sys.path.insert(0, SRC_DATASET_PATH)
else:
    print("❌ ERROR: data_pipeline.py not found in any of the possible paths.")
    if os.path.exists('/kaggle/input/route-resillence-src'):
        print("Files in '/kaggle/input/route-resillence-src':", os.listdir('/kaggle/input/route-resillence-src'))
    else:
        print("Dataset directory '/kaggle/input/route-resillence-src' is missing.")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
from torch.utils.data import DataLoader

from data_pipeline import (
    fetch_osm_roads,
    rasterize_osm_to_mask,
    tile_image_and_mask,
    add_synthetic_occlusion,
    build_binary_road_mask,
)
from segmentation_train import (
    RoadDataset,
    build_model,
    train_model,
    predict_mask,
    compute_metrics,
    get_train_augmentations,
    get_val_augmentations,
)
from geo_utils import assert_projected_crs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU not detected! Enable in Settings → Accelerator.')

In [ ]:
# ============================================================
# OSM GROUND TRUTH GENERATION (subset for alignment verification)
# ============================================================
from data_pipeline import fetch_osm_roads, rasterize_osm_to_mask
import rasterio

def load_aligned_osm_train_data(image_path: str, bbox: tuple) -> tuple:
    """Loads sat raster, fetches OSM vector roads, and rasterizes to an aligned mask."""
    with rasterio.open(image_path) as src:
        profile = src.profile
        img = src.read()
        img = np.transpose(img, (1, 2, 0)) # HWC
    gdf_osm = fetch_osm_roads(bbox, target_epsg=profile['crs'].to_epsg())
    osm_mask = rasterize_osm_to_mask(gdf_osm, profile)
    return img, osm_mask


In [ ]:
# ============================================================
# PATH SANITY CHECK
# ============================================================
import os
assert os.path.exists(SRC_DATASET_PATH), f"src path not found: {SRC_DATASET_PATH}"
assert os.path.exists(os.path.join(SRC_DATASET_PATH, 'segmentation_train.py')), "segmentation_train.py missing from src dataset"
assert os.path.exists(DEEPGLOBE_DIR), f"DeepGlobe path not found: {DEEPGLOBE_DIR}"
print("✅ All paths verified, ready to proceed.")

In [ ]:
# ============================================================
# EXPLORE DEEPGLOBE DATASET STRUCTURE
# ============================================================

print('DeepGlobe directory contents:')
for entry in sorted(os.listdir(DEEPGLOBE_DIR)):
    full = os.path.join(DEEPGLOBE_DIR, entry)
    if os.path.isdir(full):
        print(f'  {entry}/  ({len(os.listdir(full))} files)')
    else:
        print(f'  {entry}  ({os.path.getsize(full)} bytes)')

# Check class_dict.csv
class_dict_path = os.path.join(DEEPGLOBE_DIR, 'class_dict.csv')
if os.path.exists(class_dict_path):
    class_dict_df = pd.read_csv(class_dict_path)
    print('\nclass_dict.csv:')
    print(class_dict_df.to_string())
else:
    print('\n⚠️  class_dict.csv not found — check DEEPGLOBE_DIR path')

# Peek at a sample image to check tile dimensions
train_dir = os.path.join(DEEPGLOBE_DIR, 'train')
sample_sats = sorted(glob(os.path.join(train_dir, '*_sat.*')))
if sample_sats:
    sample_img = Image.open(sample_sats[0])
    print(f'\nSample tile: {os.path.basename(sample_sats[0])}')
    print(f'  Size: {sample_img.size} (W x H), Mode: {sample_img.mode}')
    print(f'  Total train sat images: {len(sample_sats)}')
else:
    print('\n⚠️  No *_sat.* files found in train/ — check dataset structure')

# Check test/ directory contents and mask availability
test_dir = os.path.join(DEEPGLOBE_DIR, 'test')
if os.path.isdir(test_dir):
    test_files = os.listdir(test_dir)
    test_masks = [f for f in test_files if '_mask' in f]
    print(f'\nTest directory: {test_dir}')
    print(f'  Total files: {len(test_files)}')
    print(f'  Mask files found: {len(test_masks)}')
    # Note: Since no masks exist in test/, it is unusable for supervised training/validation and should be ignored.
else:
    print('\nTest directory not found or is not a directory.')

In [ ]:
# FALLBACK: If no valid pairs with masks were found in valid/, split train 85/15 spatially
if not val_imgs:
    print('No validation pairs with masks found in valid/ -- splitting train 85/15 spatially...')
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.model_selection import GroupKFold
    import numpy as np
    
    # Extract visual features (color centroids/variance) as parent scene proxy
    features = []
    for img in train_imgs:
        mean = img.mean(axis=(0, 1))
        std = img.std(axis=(0, 1))
        features.append(np.concatenate([mean, std]))
    features = np.array(features)
    
    # Cluster into 15 geographic/parent-scene groups
    n_clusters = min(15, len(train_imgs))
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42)
    groups = kmeans.fit_predict(features)
    
    # Split by group
    gkf = GroupKFold(n_splits=5)
    train_idx, val_idx = next(gkf.split(train_imgs, train_msks, groups=groups))
    
    val_imgs   = [train_imgs[i] for i in val_idx]
    val_msks   = [train_msks[i] for i in val_idx]
    train_imgs = [train_imgs[i] for i in train_idx]
    train_msks = [train_msks[i] for i in train_idx]

print(f'\nFinal: {len(train_imgs)} train / {len(val_imgs)} val tiles')

# Quick sanity: show road pixel coverage
road_pcts = [m.mean()*100 for m in train_msks[:20]]
print(f'Road pixel coverage (first 20 train tiles): '
      f'mean={np.mean(road_pcts):.1f}%, range=[{min(road_pcts):.1f}, {max(road_pcts):.1f}]%')

In [ ]:
# ============================================================
# DATALOADERS & BALANCED SAMPLING
# ============================================================
from torch.utils.data import WeightedRandomSampler

train_ds = RoadDataset(
    train_imgs, train_msks,
    augmentations=get_train_augmentations(TILE_SIZE),
    apply_occlusion=True,   # synthetic occlusion for robustness
)
val_ds = RoadDataset(
    val_imgs, val_msks,
    augmentations=get_val_augmentations(),
    apply_occlusion=False,
)

# Class-balanced sampling: weight tiles by road density
weights = []
for m in train_msks:
    road_pixels = float(m.sum())
    total_pixels = float(m.size)
    weight = road_pixels / total_pixels + 0.05
    weights.append(weight)
    
sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'Train: {len(train_ds)} tiles ({len(train_loader)} batches) | '
      f'Val: {len(val_ds)} tiles ({len(val_loader)} batches)')

In [ ]:
# ============================================================
# BUILD MODEL + TRAIN
# ============================================================

model = build_model(
    architecture=ARCHITECTURE,
    encoder_name=ENCODER,
    encoder_weights='imagenet',
    in_channels=IN_CHANNELS,  # DeepGlobe = RGB = 3, confirmed
    classes=1,
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
)

In [ ]:
# ============================================================
# VISUAL SANITY CHECK — Inference on validation tiles
# ============================================================

best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.to(device)
model.eval()

NUM_EXAMPLES = min(5, len(val_imgs))
fig, axes = plt.subplots(NUM_EXAMPLES, 3, figsize=(14, 4 * NUM_EXAMPLES))
if NUM_EXAMPLES == 1:
    axes = axes[np.newaxis, :]

for i in range(NUM_EXAMPLES):
    img = val_imgs[i]
    gt  = val_msks[i]
    pred = predict_mask(model, img, device=device, threshold=0.5)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Satellite Image')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(pred, cmap='gray')
    axes[i, 1].set_title('Predicted Mask')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(gt, cmap='gray')
    axes[i, 2].set_title('Ground Truth')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/val_predictions.png', dpi=150)
plt.show()
print('Saved to /kaggle/working/val_predictions.png')

In [ ]:
# ============================================================
# FINAL METRICS SUMMARY
# ============================================================
import json

print('='*60)
print('TRAINING SUMMARY')
print('='*60)
print(f'Dataset      : DeepGlobe Road Extraction')
print(f'Architecture : {ARCHITECTURE} / {ENCODER}')
print(f'Tile size    : {TILE_SIZE}x{TILE_SIZE}')
print(f'Epochs       : {NUM_EPOCHS}')
print(f'Best Val IoU : {max(history["val_iou"]):.4f}  (epoch {np.argmax(history["val_iou"])+1})')
print(f'Best Val Dice: {max(history["val_dice"]):.4f}')
print(f'Final Train Loss : {history["train_loss"][-1]:.4f}')
print(f'Final Val Loss   : {history["val_loss"][-1]:.4f}')
print('='*60)

# Save metrics for dashboard consumption
metrics = {
    'iou': float(max(history['val_iou'])),
    'dice': float(max(history['val_dice'])),
    'best_epoch': int(np.argmax(history['val_iou']) + 1),
    'dataset': 'DeepGlobe',
    'architecture': f'{ARCHITECTURE}/{ENCODER}',
}
metrics_path = '/kaggle/working/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nMetrics saved to {metrics_path}')

# Confirm checkpoints exist
for fname in ['best_model.pth', 'final_model.pth']:
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'✓ {fname} saved ({size_mb:.1f} MB) → {fpath}')
    else:
        print(f'✗ {fname} NOT FOUND at {fpath}')

print()
print('⚠️  REMINDER: Download checkpoints + metrics.json from Output panel')
print('   before closing this session. /kaggle/working/ does NOT persist')
print('   unless you Save Version or download manually.')

In [ ]:
# ============================================================
# OCCLUSION RECALL METRICS EVALUATION
# ============================================================
from segmentation_train import compute_occlusion_recall_metrics
from data_pipeline import add_synthetic_occlusion

occluded_ious = []
occluded_dices = []
non_occluded_ious = []
non_occluded_dices = []

print("Evaluating Occlusion-Recall metrics on validation set...")
for img, gt in zip(val_imgs[:100], val_msks[:100]):  # Evaluate on up to 100 val tiles
    # Apply synthetic occlusion to image (e.g. shadow or canopy)
    occ_img, occ_mask = add_synthetic_occlusion(img, gt, occlusion_type='shadow', return_mask=True)
    
    # Run model prediction
    pred = predict_mask(model, occ_img, device=device, threshold=0.5)
    
    # Compute metrics
    scores = compute_occlusion_recall_metrics(pred, gt, occ_mask)
    
    occluded_ious.append(scores['occluded']['iou'])
    occluded_dices.append(scores['occluded']['dice'])
    non_occluded_ious.append(scores['non_occluded']['iou'])
    non_occluded_dices.append(scores['non_occluded']['dice'])

print(f"Occluded Regions     - Mean IoU: {np.mean(occluded_ious):.4f}, Mean Dice: {np.mean(occluded_dices):.4f}")
print(f"Non-Occluded Regions - Mean IoU: {np.mean(non_occluded_ious):.4f}, Mean Dice: {np.mean(non_occluded_dices):.4f}")


In [ ]:
# ============================================================
# TOPOLOGICAL ACCURACY (APL ERROR) EVALUATION
# ============================================================
from topological_accuracy import compute_apl_error
from skeleton_to_graph import mask_to_graph
import affine

apl_errors = []
print("Evaluating Topological Accuracy (APL Error) on validation set...")
for img, gt in zip(val_imgs[:10], val_msks[:10]):  # Snapping a subset of 10 validation graphs
    # Using local pixel transform as validation split has no georeferencing metadata
    pixel_transform = affine.Affine(1.0, 0.0, 0.0, 0.0, -1.0, 0.0)
    
    pred_mask = predict_mask(model, img, device=device, threshold=0.5)
    
    try:
        pred_graph = mask_to_graph(pred_mask, pixel_transform, source_crs_epsg=32643)
        gt_graph = mask_to_graph(gt, pixel_transform, source_crs_epsg=32643)
        
        mae, mpe, reach_delta = compute_apl_error(pred_graph, gt_graph, num_point_pairs=50)
        apl_errors.append(mpe)
    except Exception as e:
        continue

if apl_errors:
    print(f"Mean APL Percentage Error: {np.mean(apl_errors)*100:.2f}%")
else:
    print("Could not evaluate APL (empty graphs).")


## Next Steps

1. **Download** `best_model.pth` and `metrics.json` from the **Output** panel (right sidebar → `/kaggle/working/`).
2. Place `metrics.json` in `outputs/` for the Streamlit dashboard to consume.
3. Use `best_model.pth` to run inference → `skeleton_to_graph.py` → `graph_healing.py` → `centrality.py` → dashboard.
4. If IoU < 0.50: try `UnetPlusPlus` or `DeepLabV3Plus`, increase `NUM_EPOCHS` to 50, or reduce `TILE_SIZE` to 256.
5. **SAR training** (HybridSAR dataset): requires changing `IN_CHANNELS` and normalization — see TODO in config cell.